# 把方程、矩阵与向量读回实际问题

这份实验对应正文中的配料与双声道例子。依赖 NumPy、Matplotlib；滑块可选，未安装 ipywidgets 时直接修改函数参数。所有配料数据均为教学假设，不是营养建议。

先预测，再运行。每个数组都明确坐标顺序和单位。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 一份配方，两个要求

每 100 克甲有蛋白质 10 克、脂肪 20 克；每 100 克乙有蛋白质 20 克、脂肪 10 克。目标是蛋白质 50 克、脂肪 40 克。

`A` 的行顺序是蛋白质、脂肪；列顺序是甲、乙。用量按每 100 克为一份计数。先算一份甲加两份乙，再算三份甲加一份乙。两份配方都满足蛋白质要求吗？脂肪呢？


In [ ]:
A = np.array([[10, 20], [20, 10]])
target = np.array([50, 40])
for amounts in (np.array([1, 2]), np.array([3, 1])):
    nutrients = A @ amounts
    print('甲乙用量（克）:', 100 * amounts)
    print('蛋白质、脂肪（克）:', nutrients, '是否都符合:', np.array_equal(nutrients, target))


### 沿蛋白质约束走，观察脂肪怎样变化

由 `10x+20y=50` 得到 `y=2.5−0.5x`。令甲用量 `x` 从 0 变到 5，乙跟着这个式子变，蛋白质始终不变。

先预测：脂肪会随 `x` 增加还是减少？在什么位置恰好为 40 克？左图中的点沿第一条约束移动；只有到第二条约束上，两项要求才都满足。


In [ ]:
def inspect_recipe(x=1):
    y = 2.5 - 0.5 * x
    amounts = np.array([x, y])
    nutrients = A @ amounts
    grid = np.linspace(0, 5, 251)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), dpi=140)
    ax = axes[0]
    ax.plot(grid, 2.5 - 0.5 * grid, color='#2878b5', label='Protein = 50 g')
    ax.plot(grid, 4 - 2 * grid, color='#d97922', label='Fat = 40 g')
    ax.scatter(x, y, color='#c53351', s=55, zorder=5)
    ax.annotate('({:g}, {:g})'.format(x, y), (x, y),
                xytext=(8, 8), textcoords='offset points', color='#c53351')
    ax.set(xlim=(-0.2, 5.2), ylim=(-0.2, 4.2),
           xlabel='A amount (100 g units)', ylabel='B amount (100 g units)',
           title='A recipe must satisfy both constraints')
    ax.axhline(0, color='0.65', lw=0.8)
    ax.axvline(0, color='0.65', lw=0.8)
    ax.grid(alpha=0.2)
    ax.legend()
    positions = np.arange(2)
    axes[1].bar(positions - 0.18, nutrients, width=0.36,
                color=['#2878b5', '#d97922'], label='Recipe')
    axes[1].bar(positions + 0.18, target, width=0.36,
                facecolor='none', edgecolor='#359969', linewidth=2, label='Target')
    axes[1].set(xticks=positions, xticklabels=['Protein', 'Fat'],
                ylim=(0, 110), ylabel='grams', title='Actual amounts vs requirements')
    axes[1].legend()
    fig.tight_layout()
    plt.show()
    print('甲、乙（克）:', np.round(100 * amounts, 2))
    print('蛋白质、脂肪（克）:', np.round(nutrients, 2))
    print('实际减目标（克）:', np.round(nutrients - target, 2))


In [ ]:
inspect_recipe(x=3)


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print('直接修改 inspect_recipe(x=...)，x 范围是 0 到 5。')
else:
    display(widgets.interactive(
        inspect_recipe,
        x=widgets.FloatSlider(value=3, min=0, max=5, step=0.1,
                              description='A amount', continuous_update=False)))


### 用消元确认交点，而不是靠图估计

第二式减去第一式的两倍：`−30y=−60`。自己求出 `y`，再代回第一式求 `x`。代码仅把这几步写出来，没有调用求解器。


In [ ]:
y = (40 - 2 * 50) / (10 - 2 * 20)
x = (50 - 20 * y) / 10
print('x,y =', x, y, '对应原料克数:', 100 * x, 100 * y)
assert np.allclose(A @ np.array([x, y]), target)
inspect_recipe(x=x)


### 实数解是否总是可用配方？

把目标改为蛋白质 10 克、脂肪 0 克。纸上求解后再运行。若有负量原料，矩阵计算仍可能相等，但不能拿这个数当实际配方。


In [ ]:
amounts = np.array([-1/3, 2/3])
print('用量（每 100 克为一份）:', amounts)
print('得到的两项成分:', A @ amounts)
print('每种用量都非负吗？', bool(np.all(amounts >= 0)))
assert np.allclose(A @ amounts, [10, 0])


## 同一段双声道，换一组基来记录

`L,R` 是同一时刻左、右信号采样值。约定 `M=(L+R)/2,S=(L−R)/2`。先手算 `(L,R)=(3,1)` 的新坐标，再从新坐标重建旧坐标。

下面的 3、1 是任意刻度的数值示例，不表示某种设备的允许输入幅度。


In [ ]:
L, R = 3., 1.
M, S = (L + R) / 2, (L - R) / 2
basis = np.array([[1, 1], [1, -1]])
reconstructed = basis @ np.array([M, S])
print('M,S:', M, S, '重建 L,R:', reconstructed)
assert np.allclose(reconstructed, [L, R])


### 对一整段信号做同样的计算

下面构造一段共同波形，加上一段左右相反的波形。先预测：`M` 会保留哪段？`S` 会保留哪段？然后运行看三张图。最后一张把共同部分保留、左右差别丢掉，观察原信号还能否恢复。


In [ ]:
time = np.linspace(0, 1, 501)
common = 0.4 * np.sin(2 * np.pi * 3 * time)
difference = 0.2 * np.sin(2 * np.pi * 7 * time)
left, right = common + difference, common - difference
mid, side = (left + right) / 2, (left - right) / 2
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True, dpi=140)
series = [
    (left, right, 'L', 'R', 'Original left and right'),
    (mid, side, 'M', 'S', 'Same signal in a different basis'),
    (left, mid, 'Original L', 'L after dropping S', 'Dropping a coordinate loses information'),
]
for ax, (first, second, label1, label2, title) in zip(axes, series):
    ax.plot(time, first, color='#2878b5', label=label1)
    ax.plot(time, second, color='#d97922', linestyle='--', label=label2)
    ax.set(ylim=(-0.65, 0.65), ylabel='sample value', title=title)
    ax.grid(alpha=0.2)
    ax.legend(loc='upper right')
axes[-1].set_xlabel('time (seconds)')
fig.tight_layout()
plt.show()
assert np.allclose(mid, common)
assert np.allclose(side, difference)
assert np.allclose(mid + side, left)
assert np.allclose(mid - side, right)
print('保留 M 和 S 可以重建两条波形；只留 M 一般不行。')


## 从问题重新讲一遍

1. 配料矩阵的一行、一列、输入向量、输出向量各代表什么，单位是什么？
2. 为什么只满足蛋白质要求的配方不止一个，同时满足两项要求才需要找交集？
3. 为什么负数在实际配料量里不允许，在声音采样值里却很自然？
4. 从 `(L,R)` 改记 `(M,S)`，为什么不丢信息？丢掉 `S` 后，为什么无法完整恢复？
